# ML Pipeline — Feature Engineering → Train → Register → Predict

A single notebook that runs the full ML pipeline: engineers time-series features from Gold layer data and publishes to the Databricks Feature Store, then reads features back, trains a Linear Regression model with MLflow tracking, registers the model to Unity Catalog, and makes predictions on test data. Uses OOP class-based design with Python `logging` for structured output. All parameters come from `dev.yaml`.

**Pipeline:**

```
Gold  →  Feature Engineering  →  Feature Store  →  Train + Log  →  Register Model  →  Predict
```

In [0]:
%pip install databricks-feature-engineering

dbutils.library.restartPython()

In [0]:
import yaml
import logging
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from databricks.feature_engineering import FeatureEngineeringClient

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)


class BaseLayer:
    """Base class for ML pipeline stages with shared config and logging.

    Provides config loading from a YAML file, logger setup, and schema creation
    in Unity Catalog. Subclasses implement stage-specific logic.

    Attributes:
        config_path: Path to the YAML configuration file.
        layer_name: Name of the stage (used as the logger name).
        cfg: Parsed YAML configuration dictionary.
        catalog: Unity Catalog name from config.
    """

    def __init__(self, config_path: str, layer_name: str):
        self.config_path = config_path
        self.layer_name = layer_name
        self.logger = logging.getLogger(layer_name)
        self._load_config()

    def _load_config(self) -> None:
        """Load YAML configuration from the configured path."""
        with open(self.config_path, "r") as f:
            self.cfg = yaml.safe_load(f)
        self.catalog = self.cfg["catalog"]
        self.logger.info(f"Config loaded from {self.config_path}")

    def ensure_schema(self, schema_name: str) -> None:
        """Create the target schema in Unity Catalog if it doesn't exist."""
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {self.catalog}.{schema_name}")
        self.logger.info(f"Ensured schema exists: {self.catalog}.{schema_name}")


class FeatureEngineering(BaseLayer):
    """Feature Engineering layer: transform Gold data into ML-ready features for the Feature Store.

    Reads the Gold layer table, engineers lag features, rolling window features,
    and calendar features, then writes the result to the Databricks Feature Store
    as a time series feature table.

    Attributes:
        gold_full: Fully qualified Gold source table name.
        feature_full: Fully qualified Feature Store target table name.
        primary_keys: List of primary key columns for the feature table.
        timeseries_column: Name of the timeseries column.
    """

    def __init__(self, config_path: str = "/Workspace/Users/girija.zirange@bizmetric.com/MLE/config/dev.yaml"):
        super().__init__(config_path, "feature_engineering")
        self.layer_cfg = self.cfg["feature_engineering"]
        self.gold_schema = self.cfg["schemas"]["gold"]
        self.features_schema = self.cfg["schemas"]["features"]
        self.gold_table = self.layer_cfg["source"]["gold_table"]
        self.feature_table = self.layer_cfg["target"]["feature_table"]
        self.gold_full = f"{self.catalog}.{self.gold_schema}.{self.gold_table}"
        self.feature_full = f"{self.catalog}.{self.features_schema}.{self.feature_table}"
        self.primary_keys = self.layer_cfg["primary_keys"]
        self.timeseries_column = self.layer_cfg["timeseries_column"]
        self.fe = FeatureEngineeringClient()
        self.logger.info(f"Gold source:      {self.gold_full}")
        self.logger.info(f"Feature target:   {self.feature_full}")

    def read_gold(self):
        """Read the Gold Delta table (aggregated product-city-date demand)."""
        self.gold_df = spark.table(self.gold_full)
        self.gold_count = self.gold_df.count()
        self.logger.info(f"Gold rows read: {self.gold_count:,}")
        return self.gold_df

    def create_time_features(self, df):
        """Add calendar/time features derived from transaction_date."""
        self.logger.info("Adding calendar/time features...")
        df = (
            df
            .withColumn("day_of_week", F.dayofweek("transaction_date"))
            .withColumn("day_of_month", F.dayofmonth("transaction_date"))
            .withColumn("is_weekend", F.dayofweek("transaction_date").isin(1, 7))
            .withColumn("quarter", F.quarter("transaction_date"))
        )
        self.logger.info(f"Columns after time features: {len(df.columns)}")
        return df

    def create_lag_features(self, df):
        """Add lag features partitioned by (product_name, destination_city), ordered by transaction_date."""
        self.logger.info("Adding lag features...")
        lag_windows = self.layer_cfg["features"]["lag_windows"]
        lag_columns = self.layer_cfg["features"]["lag_columns"]
        w = Window.partitionBy("product_name", "destination_city").orderBy("transaction_date")

        for col_name in lag_columns:
            for lag_n in lag_windows:
                if col_name == "demand_quantity":
                    feat_name = f"lag_{lag_n}_demand"
                else:
                    feat_name = f"lag_{lag_n}_{col_name}"
                df = df.withColumn(feat_name, F.lag(col_name, lag_n).over(w))
                self.logger.info(f"  Created {feat_name}")

        self.logger.info(f"Columns after lag features: {len(df.columns)}")
        return df

    def create_rolling_features(self, df):
        """Add rolling window features (7-row avg/sum/max) partitioned by product-city."""
        self.logger.info("Adding rolling window features...")
        rolling_window = self.layer_cfg["features"]["rolling_window"]
        rolling_columns = self.layer_cfg["features"]["rolling_columns"]
        w = (
            Window
            .partitionBy("product_name", "destination_city")
            .orderBy("transaction_date")
            .rowsBetween(-(rolling_window - 1), 0)
        )

        for col_name in rolling_columns:
            if col_name == "demand_quantity":
                df = (
                    df
                    .withColumn(f"rolling_{rolling_window}_avg_demand", F.avg(col_name).over(w))
                    .withColumn(f"rolling_{rolling_window}_sum_demand", F.sum(col_name).over(w))
                    .withColumn(f"rolling_{rolling_window}_max_demand", F.max(col_name).over(w))
                )
                self.logger.info(f"  Created rolling_{rolling_window}_avg/sum/max_demand")
            else:
                df = df.withColumn(f"rolling_{rolling_window}_avg_{col_name}", F.avg(col_name).over(w))
                self.logger.info(f"  Created rolling_{rolling_window}_avg_{col_name}")

        self.logger.info(f"Columns after rolling features: {len(df.columns)}")
        return df

    def assemble(self):
        """Assemble the final feature DataFrame with a deterministic column order."""
        self.logger.info("Assembling final feature DataFrame...")

        pk_cols = ["product_name", "destination_city", "transaction_date"]
        original_cols = [
            "transaction_month", "product_category", "quantity_unit",
            "demand_quantity",
            "avg_unit_price_usd", "avg_available_inventory",
        ]
        engineered_cols = [
            "lag_1_demand", "lag_7_demand",
            "rolling_7_avg_demand", "rolling_7_sum_demand",
            "rolling_7_max_demand",
            "day_of_week", "day_of_month", "is_weekend", "quarter",
        ]
        final_cols = pk_cols + original_cols + engineered_cols
        self.feature_df = self.feature_df.select(*final_cols)
        self.feature_count = self.feature_df.count()
        self.logger.info(f"Final feature rows:    {self.feature_count:,}")
        self.logger.info(f"Final feature columns: {len(self.feature_df.columns)}")
        return self.feature_df

    def write_feature_store(self) -> None:
        """Write the feature DataFrame to the Databricks Feature Store as a time series feature table."""
        self.logger.info("Writing to Feature Store...")
        self.ensure_schema(self.features_schema)

        table_exists = spark.catalog.tableExists(self.feature_full)
        if table_exists:
            self.logger.info("Feature table exists — dropping and recreating...")
            spark.sql(f"DROP TABLE IF EXISTS {self.feature_full}")

        self.fe.create_table(
            name=self.feature_full,
            primary_keys=self.primary_keys,
            timeseries_column=self.timeseries_column,
            df=self.feature_df,
            schema=self.feature_df.schema,
            description=(
                "Product-city demand features for forecasting: lag, rolling window, "
                "and calendar features. Time series feature table keyed by "
                "(product_name, destination_city, transaction_date)."
            ),
        )
        self.logger.info(f"Created feature table: {self.feature_full}")
        self.logger.info(f"  Rows written: {self.feature_count:,}")

    def validate(self) -> None:
        """Log feature engineering summary."""
        self.logger.info(f"Features: {self.feature_count:,} rows, {len(self.feature_df.columns)} columns → {self.feature_full}")

    def run(self) -> None:
        """Execute the full feature engineering pipeline."""
        self.read_gold()
        self.feature_df = self.gold_df
        self.feature_df = self.create_time_features(self.feature_df)
        self.feature_df = self.create_lag_features(self.feature_df)
        self.feature_df = self.create_rolling_features(self.feature_df)
        self.assemble()
        self.write_feature_store()
        self.validate()

In [0]:
import mlflow
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from mlflow.models import infer_signature


class ModelTraining(BaseLayer):
    """Training layer: read features, train model, register to Unity Catalog, and predict.

    Reads the engineered feature table from the Feature Store, splits
    chronologically into train/val/test, trains a Linear Regression model
    with an sklearn preprocessing pipeline, logs the experiment to MLflow,
    registers the model to Unity Catalog, and makes predictions on test data.

    Attributes:
        feature_full: Fully qualified feature table name.
        experiment_name: MLflow experiment name.
        model_name: Unity Catalog registered model name.
    """

    def __init__(self, config_path: str = "/Workspace/Users/girija.zirange@bizmetric.com/MLE/config/dev.yaml"):
        super().__init__(config_path, "model_training")
        self.fe_cfg = self.cfg["feature_engineering"]
        self.features_schema = self.cfg["schemas"]["features"]
        self.feature_table = self.fe_cfg["target"]["feature_table"]
        self.feature_full = f"{self.catalog}.{self.features_schema}.{self.feature_table}"
        self.experiment_name = "/Users/girija.zirange@bizmetric.com/MLE/demand_forecasting"
        self.model_name = f"{self.catalog}.models.demand_forecast_lr"
        self.prediction_table = f"{self.catalog}.models.demand_predictions"
        self.target = "demand_quantity"
        self.categorical_features = ["product_name", "destination_city", "product_category", "quantity_unit"]
        self.logger.info(f"Feature table:    {self.feature_full}")
        self.logger.info(f"Experiment:       {self.experiment_name}")
        self.logger.info(f"Model name:       {self.model_name}")
        self.logger.info(f"Prediction table: {self.prediction_table}")

    def read_features(self):
        """Read the feature table from Feature Store and prepare for training."""
        self.df = spark.table(self.feature_full).toPandas()
        self.logger.info(f"Features: {self.df.shape[0]:,} rows, {self.df.shape[1]} columns")

        if "is_weekend" in self.df.columns:
            self.df["is_weekend"] = self.df["is_weekend"].astype(int)

        self.df["transaction_date"] = pd.to_datetime(self.df["transaction_date"])
        self.df = self.df.sort_values("transaction_date").reset_index(drop=True)

        exclude = ["transaction_date", self.target]
        self.numerical_features = [
            c for c in self.df.columns
            if c not in exclude + self.categorical_features
        ]
        self.feature_cols = self.categorical_features + self.numerical_features
        self.logger.info(f"Target: {self.target} | Features: {len(self.feature_cols)} ({len(self.categorical_features)} cat, {len(self.numerical_features)} num)")

    def split_data(self):
        """Chronological split into train (64%), validation (16%), and test (20%)."""
        n = len(self.df)
        idx_test_start = int(n * 0.80)
        idx_val_start = int(idx_test_start * 0.80)

        self.X_train = self.df.iloc[:idx_val_start][self.feature_cols]
        self.y_train = self.df.iloc[:idx_val_start][self.target]
        self.X_val = self.df.iloc[idx_val_start:idx_test_start][self.feature_cols]
        self.y_val = self.df.iloc[idx_val_start:idx_test_start][self.target]
        self.X_test = self.df.iloc[idx_test_start:][self.feature_cols]
        self.y_test = self.df.iloc[idx_test_start:][self.target]

        self.logger.info(f"Train: {len(self.X_train):,} | Val: {len(self.X_val):,} | Test: {len(self.X_test):,}")

    def train_and_log(self):
        """Build sklearn pipeline, train model, and log experiment to MLflow."""
        preprocessor = ColumnTransformer(
            transformers=[
                ("num", Pipeline([
                    ("imputer", SimpleImputer(strategy="mean")),
                    ("scaler", StandardScaler()),
                ]), self.numerical_features),
                ("cat", Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]), self.categorical_features),
            ]
        )

        self.model = Pipeline([
            ("preprocessor", preprocessor),
            ("regressor", LinearRegression()),
        ])

        mlflow.set_experiment(self.experiment_name)

        def eval_metrics(y_true, y_pred):
            """Compute RMSE, MAE, R², and MAPE for regression predictions."""
            rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
            mae = float(mean_absolute_error(y_true, y_pred))
            r2 = float(r2_score(y_true, y_pred))
            mask = y_true != 0
            mape = float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100) if mask.any() else float("inf")
            return {"rmse": rmse, "mae": mae, "r2": r2, "mape": mape}

        with mlflow.start_run(run_name="linear_regression_baseline") as run:
            self.model.fit(self.X_train, self.y_train)

            y_train_pred = self.model.predict(self.X_train)
            y_val_pred = self.model.predict(self.X_val)
            y_test_pred = self.model.predict(self.X_test)

            train_m = eval_metrics(self.y_train, y_train_pred)
            val_m = eval_metrics(self.y_val, y_val_pred)
            test_m = eval_metrics(self.y_test, y_test_pred)

            mlflow.log_param("model_type", "LinearRegression")
            mlflow.log_param("split_strategy", "chronological")
            mlflow.log_param("train_size", len(self.X_train))
            mlflow.log_param("val_size", len(self.X_val))
            mlflow.log_param("test_size", len(self.X_test))
            mlflow.log_param("n_features", len(self.feature_cols))

            for prefix, m in [("train", train_m), ("val", val_m), ("test", test_m)]:
                for name, val in m.items():
                    mlflow.log_metric(f"{prefix}_{name}", val)

            signature = infer_signature(self.X_train.head(100), self.model.predict(self.X_train.head(100)))
            model_info = mlflow.sklearn.log_model(
                self.model,
                name="model",
                signature=signature,
                input_example=self.X_train.head(3),
            )

            self.run_id = run.info.run_id
            self.model_uri = model_info.model_uri

            self.logger.info(f"MLflow Run: {self.run_id}")
            self.logger.info(f"R² — Train: {train_m['r2']:.4f} | Val: {val_m['r2']:.4f} | Test: {test_m['r2']:.4f}")
            self.logger.info(f"Test RMSE: {test_m['rmse']:.2f} | MAPE: {test_m['mape']:.2f}%")

    def register_model(self):
        """Register the trained model to Unity Catalog."""
        self.ensure_schema("models")
        mv = mlflow.register_model(
            model_uri=self.model_uri,
            name=self.model_name,
        )
        self.model_version = mv.version
        self.logger.info(f"Registered: {self.model_name} (version {self.model_version})")

    def predict(self):
        """Load the registered model, make predictions on the test set, and write to a table."""
        self.logger.info("Loading registered model for predictions...")
        loaded_model = mlflow.pyfunc.load_model(f"models:/{self.model_name}/{self.model_version}")
        predictions = loaded_model.predict(self.X_test)

        # Get identifiers for the test set rows
        n = len(self.df)
        idx_test_start = int(n * 0.80)
        test_ids = self.df.iloc[idx_test_start:][["product_name", "destination_city", "transaction_date"]].reset_index(drop=True)

        self.pred_df = pd.DataFrame({
            "product_name": test_ids["product_name"].values,
            "destination_city": test_ids["destination_city"].values,
            "transaction_date": test_ids["transaction_date"].values,
            "actual_demand": self.y_test.values,
            "predicted_demand": predictions,
            "model_name": self.model_name,
            "model_version": str(self.model_version),
            "mlflow_run_id": self.run_id,
        })

        # Write all predictions to a Unity Catalog table
        pred_spark_df = spark.createDataFrame(self.pred_df)
        self.ensure_schema("models")
        (pred_spark_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(self.prediction_table))
        self.logger.info(f"Written {len(self.pred_df):,} predictions to {self.prediction_table}")

    def run(self) -> None:
        """Execute the full training pipeline."""
        self.read_features()
        self.split_data()
        self.train_and_log()
        self.register_model()
        self.predict()

In [0]:
# Run the full ML pipeline: Feature Engineering → Train → Register → Predict
fe = FeatureEngineering()
fe.run()

trainer = ModelTraining()
trainer.run()

logging.getLogger("pipeline").info("ML pipeline complete: Feature Engineering → Train → Register → Predict")